# 02 — Missing values

**Child Mind Institute — Problematic Internet Use**

Цель ноутбука — понять не только количество пропусков, но и их структуру.

Разбираем:

- missingness по отдельным признакам;
- missingness по участникам;
- различия между участниками с известным и неизвестным `sii`;
- missingness внутри классов `sii`;
- отсутствие целых инструментов / обследований;
- структурные пропуски, связанные с возрастом.



## 1. Imports and data loading

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

TARGET = "sii"
ID_COL = "id"
AGE_COL = "Basic_Demos-Age"


In [ ]:
# Работает при запуске Jupyter как из корня проекта, так и из project/notebooks/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
DATA_DICT_PATH = DATA_DIR / "data_dictionary.csv"

for path in [TRAIN_PATH, TEST_PATH, DATA_DICT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
data_dict = pd.read_csv(DATA_DICT_PATH)

MODEL_FEATURES = [c for c in test.columns if c != ID_COL]
train_labeled = train.loc[train[TARGET].notna()].copy()
train_unlabeled = train.loc[train[TARGET].isna()].copy()

print("Train:", train.shape)
print("Test:", test.shape)
print("Labeled train:", train_labeled.shape)
print("Unlabeled train:", train_unlabeled.shape)
print(f"Missing target: {train[TARGET].isna().mean():.1%}")


## 2. Missing target

`train.csv` содержит строки без `sii`. Они могут быть полезны для unsupervised/statistical анализа признаков, но не входят в обычное supervised-обучение с `sii` в качестве target.

Поэтому далее важно различать:

- `train` — все участники;
- `train_labeled` — только участники с известным `sii`;
- `train_unlabeled` — участники без `sii`.


In [ ]:
target_availability = pd.DataFrame({
    "group": ["sii available", "sii missing"],
    "n": [train[TARGET].notna().sum(), train[TARGET].isna().sum()],
    "fraction": [train[TARGET].notna().mean(), train[TARGET].isna().mean()],
})

display(target_availability)


## 3. Missingness by feature

In [ ]:
feature_missingness = pd.DataFrame({
    "train_missing_n": train[MODEL_FEATURES].isna().sum(),
    "train_missing_%": train[MODEL_FEATURES].isna().mean() * 100,
    "test_missing_n": test[MODEL_FEATURES].isna().sum(),
    "test_missing_%": test[MODEL_FEATURES].isna().mean() * 100,
})

feature_missingness["train_available_n"] = (
    len(train) - feature_missingness["train_missing_n"]
)
feature_missingness["test_available_n"] = (
    len(test) - feature_missingness["test_missing_n"]
)
feature_missingness["test_minus_train_pp"] = (
    feature_missingness["test_missing_%"]
    - feature_missingness["train_missing_%"]
)

feature_missingness = feature_missingness.sort_values(
    "train_missing_%",
    ascending=False,
)

display(feature_missingness.round(2))


### Most sparse features

В supplied `test.csv` очень мало строк, поэтому `test_missing_%` имеет высокую дискретность и не должен интерпретироваться как надёжная оценка distribution shift. Здесь test используется в основном для контроля схемы.


In [ ]:
top_missing = feature_missingness.loc[
    feature_missingness["train_missing_%"] > 0,
    "train_missing_%",
].head(30).sort_values()

plt.figure(figsize=(10, 9))
plt.barh(top_missing.index, top_missing.values)
plt.xlabel("Missing values, %")
plt.ylabel("")
plt.title("Top 30 model features by missingness — train")
plt.tight_layout()
plt.show()


## 4. Missingness per participant

In [ ]:
participant_missingness = pd.DataFrame({
    ID_COL: train[ID_COL],
    TARGET: train[TARGET],
})

participant_missingness["n_missing"] = (
    train[MODEL_FEATURES].isna().sum(axis=1)
)
participant_missingness["missing_%"] = (
    train[MODEL_FEATURES].isna().mean(axis=1) * 100
)
participant_missingness["target_available"] = train[TARGET].notna()

display(
    participant_missingness[["n_missing", "missing_%"]]
    .describe()
    .round(2)
)


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(
    participant_missingness["n_missing"],
    bins=30,
    edgecolor="black",
)
plt.xlabel("Number of missing model features")
plt.ylabel("Participants")
plt.title("Missing features per participant")
plt.tight_layout()
plt.show()


## 5. Is missingness related to target availability?

Это критически важная проверка. Если строки без `sii` одновременно имеют существенно меньше доступных признаков, то `sii` missingness не является нейтральным относительно качества обследования.


In [ ]:
missingness_by_target_availability = (
    participant_missingness
    .groupby("target_available")["n_missing"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .rename(index={False: "sii missing", True: "sii available"})
)

display(missingness_by_target_availability.round(2))


In [ ]:
available = participant_missingness.loc[
    participant_missingness["target_available"], "n_missing"
]
missing = participant_missingness.loc[
    ~participant_missingness["target_available"], "n_missing"
]

plt.figure(figsize=(7, 4))
plt.boxplot(
    [available, missing],
    labels=["sii available", "sii missing"],
    showfliers=False,
)
plt.ylabel("Number of missing model features")
plt.title("Feature missingness vs target availability")
plt.tight_layout()
plt.show()


### Dataset-specific observation

Для исходного competition train эта проверка показывает сильное различие: строки без `sii` в среднем значительно более неполные, чем строки с известным target.

Это означает, что перед supervised-моделированием недостаточно просто удалить строки с `sii = NaN` и забыть о них: нужно помнить, что labeled subset является более полно обследованной частью исходной выборки.


## 6. Missingness within SII classes

In [ ]:
missingness_by_sii = (
    participant_missingness
    .dropna(subset=[TARGET])
    .groupby(TARGET)["n_missing"]
    .agg(["count", "mean", "median", "std"])
)

display(missingness_by_sii.round(2))


In [ ]:
sii_groups = [
    participant_missingness.loc[
        participant_missingness[TARGET] == cls,
        "n_missing",
    ]
    for cls in sorted(train_labeled[TARGET].unique())
]

sii_labels = [
    str(int(cls))
    for cls in sorted(train_labeled[TARGET].unique())
]

plt.figure(figsize=(7, 4))
plt.boxplot(
    sii_groups,
    labels=sii_labels,
    showfliers=False,
)
plt.xlabel("SII")
plt.ylabel("Number of missing model features")
plt.title("Feature missingness across SII classes")
plt.tight_layout()
plt.show()


## 7. Missingness by instrument

Отдельные колонки относятся к одному и тому же обследованию. Поэтому полезнее дополнительно смотреть missingness на уровне **инструмента**, а не только отдельных признаков.

Для каждого инструмента считаем:

- среднюю долю NaN среди его model features;
- долю участников, у которых есть хотя бы одно значение;
- долю участников, у которых заполнены все признаки этого инструмента.


In [ ]:
field_to_instrument = dict(
    zip(data_dict["Field"], data_dict["Instrument"])
)

instrument_rows = []

for instrument in data_dict["Instrument"].dropna().unique():
    cols = [
        c for c in MODEL_FEATURES
        if field_to_instrument.get(c) == instrument
    ]

    if not cols:
        continue

    instrument_rows.append({
        "instrument": instrument,
        "n_features": len(cols),
        "mean_missing_%": (
            train[cols].isna().mean().mean() * 100
        ),
        "participants_with_any_data_%": (
            train[cols].notna().any(axis=1).mean() * 100
        ),
        "participants_with_complete_data_%": (
            train[cols].notna().all(axis=1).mean() * 100
        ),
    })

instrument_missingness = (
    pd.DataFrame(instrument_rows)
    .sort_values("mean_missing_%", ascending=False)
    .reset_index(drop=True)
)

display(instrument_missingness.round(2))


In [ ]:
plot_data = (
    instrument_missingness
    .set_index("instrument")["participants_with_any_data_%"]
    .sort_values()
)

plt.figure(figsize=(10, 6))
plt.barh(plot_data.index, plot_data.values)
plt.xlabel("Participants with any data, %")
plt.ylabel("")
plt.title("Instrument availability in train")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()


## 8. Structural missingness by age

Высокий процент NaN не обязательно означает плохое качество данных.

Особенно показательные признаки:

- `PAQ_C-PAQ_C_Total` — questionnaire for children;
- `PAQ_A-PAQ_A_Total` — questionnaire for adolescents;
- `Fitness_Endurance-Max_Stage` — endurance assessment.

Если availability резко меняется с возрастом, такой missingness следует рассматривать как **структурный**.


In [ ]:
structural_cols = [
    "PAQ_C-PAQ_C_Total",
    "PAQ_A-PAQ_A_Total",
    "Fitness_Endurance-Max_Stage",
]
structural_cols = [
    c for c in structural_cols
    if c in train.columns
]

availability_by_age = pd.DataFrame({
    col: train.groupby(AGE_COL)[col].apply(
        lambda x: x.notna().mean()
    )
    for col in structural_cols
})

display(availability_by_age.round(3))


In [ ]:
plt.figure(figsize=(10, 5))

for col in availability_by_age.columns:
    plt.plot(
        availability_by_age.index,
        availability_by_age[col],
        marker="o",
        label=col,
    )

plt.xlabel("Age")
plt.ylabel("Fraction with available measurement")
plt.title("Feature availability by age")
plt.ylim(-0.05, 1.05)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 9. Target availability by age

Проверяем также, одинаково ли часто доступен `sii` в разных возрастах. Это важно, потому что supervised subset формируется неслучайным удалением строк.


In [ ]:
target_availability_by_age = (
    train
    .groupby(AGE_COL)[TARGET]
    .apply(lambda x: x.notna().mean())
    .rename("sii_available_fraction")
    .to_frame()
)

target_availability_by_age["n_participants"] = (
    train.groupby(AGE_COL).size()
)

display(target_availability_by_age.round(3))


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(
    target_availability_by_age.index,
    target_availability_by_age["sii_available_fraction"],
    marker="o",
)
plt.xlabel("Age")
plt.ylabel("Fraction with available SII")
plt.title("Target availability by age")
plt.ylim(0, 1)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 10. Compact summary tables

Эти таблицы удобно использовать позже при выборе preprocessing strategy.


In [ ]:
high_missing_features = feature_missingness.loc[
    feature_missingness["train_missing_%"] >= 50
].copy()

print(
    f"Features with >=50% missing values: "
    f"{len(high_missing_features)} / {len(MODEL_FEATURES)}"
)
display(high_missing_features.round(2))


In [ ]:
availability_bins = pd.cut(
    feature_missingness["train_missing_%"],
    bins=[-0.01, 10, 25, 50, 75, 100],
    labels=[
        "0–10%",
        "10–25%",
        "25–50%",
        "50–75%",
        "75–100%",
    ],
)

missingness_bins = (
    availability_bins
    .value_counts()
    .sort_index()
    .rename("n_features")
    .to_frame()
)

display(missingness_bins)


## 11. Main findings to carry forward


1. **Target missingness нельзя смешивать с feature missingness.** Строки без `sii` не подходят для обычного supervised training.
2. **Labeled и unlabeled subsets отличаются по полноте обследования.** Это потенциальный selection effect.
3. **Missingness имеет блочную структуру:** часто отсутствует не отдельная колонка, а значительная часть одного инструмента.
4. **PAQ missingness структурно зависит от возраста.** Простая глобальная median imputation не отражает причину таких NaN.
5. При preprocessing стоит рассмотреть:
   - missing indicators;
   - imputation внутри логически однородных feature groups;
   - модели, естественно работающие с NaN;
   - явный учёт возраста для age-dependent measurements.
6. Маленький supplied `test.csv` не позволяет надёжно оценивать train/test missingness shift.

